In [ ]:
# CNN Model

import cv2
import mediapipe
from cvzone.HandTrackingModule import HandDetector
import os
import numpy as np
import math

# Video Capture
cap= cv2.VideoCapture(0)

# Resize the frame 
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 960)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

detector= HandDetector(maxHands=1)

offset=20
imgSize=300

while True:
    success, img= cap.read()
    hands, img= detector.findHands(img)
    cv2.imshow("Image", img)

    if hands:
        hand = hands[0]
        x, y, w, h = hand['bbox']

        imgWhite = np.ones((imgSize, imgSize, 3), np.uint8) * 255

        h_img, w_img, _ = img.shape

        x1 = max(0, x - offset)
        y1 = max(0, y - offset)
        x2 = min(w_img, x + w + offset)
        y2 = min(h_img, y + h + offset)

        imgCrop = img[y1:y2, x1:x2]

        if imgCrop.size == 0:
            continue

        aspectRatio = h / w

        if aspectRatio > 1:
            k = imgSize / h
            wCal = math.ceil(k * w)
            imgResize = cv2.resize(imgCrop, (wCal, imgSize))
            wGap = math.ceil((imgSize - wCal) / 2)
            imgWhite[:, wGap:wGap + wCal] = imgResize

        else:
            k = imgSize / w
            hCal = math.ceil(k * h)
            imgResize = cv2.resize(imgCrop, (imgSize, hCal))
            hGap = math.ceil((imgSize - hCal) / 2)
            imgWhite[hGap:hGap + hCal, :] = imgResize

        cv2.imshow("Image Crop", imgCrop)
        cv2.imshow("Image White", imgWhite)
        
    if cv2.waitKey(1)==27: 
        break



cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
from cvzone.HandTrackingModule import HandDetector
import numpy as np
import os

cap = cv2.VideoCapture(0)
detector = HandDetector(maxHands=1)

sequence = []
sequence_length = 30
collecting = False
current_label = None

labels = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ") + list("0123456789")

# Create folders once
for label in labels:
    os.makedirs(f"data/raw/{label}", exist_ok=True)

while True:
    success, img = cap.read()
    hands, img = detector.findHands(img)

    key = cv2.waitKey(1) & 0xFF

    if hands:
        hand = hands[0]
        lmList = hand['lmList']

        feature_vector = []
        for lm in lmList:
            feature_vector.extend(lm)

        feature_vector = np.array(feature_vector)

        # Start collecting
        if collecting:
            sequence.append(feature_vector)

            if len(sequence) == sequence_length:
                save_path = f"data/raw/{current_label}/{np.random.randint(100000)}.npy"
                np.save(save_path, sequence)
                print(f"Saved: {save_path}")

                sequence = []
                collecting = False

    # Key controls
    if 65 <= key <= 90:  # A-Z
        current_label = chr(key)
        print(f"Selected label: {current_label}")

    elif 48 <= key <= 57:  # 0-9
        current_label = chr(key)
        print(f"Selected label: {current_label}")

    elif key == ord('s') and current_label is not None:
        collecting = True
        sequence = []
        print(f"Collecting for: {current_label}")

    elif key == 27:  # ESC
        break

    cv2.imshow("Image", img)

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
from cvzone.HandTrackingModule import HandDetector
import numpy as np
import os

# =========================
# [1] Video Capture Setup
# =========================
cap = cv2.VideoCapture(0)

# =========================
# [2] Hand Detector
# =========================
detector = HandDetector(maxHands=1)

# =========================
# [3] Dataset Settings
# =========================
sequence_length = 30
sequence = []

collecting = False
current_label = None

# 🔥 Mode control (IMPORTANT)
mode = "alphabet"   # change to "number" when needed

# =========================
# [4] Define Labels
# =========================
if mode == "alphabet":
    labels = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
else:
    labels = list("0123456789")

# =========================
# [5] Create Folder Structure
# =========================
for label in labels:
    os.makedirs(f"data/raw/{mode}/{label}", exist_ok=True)

# =========================
# [6] Main Loop
# =========================
while True:
    success, img = cap.read()
    if not success:
        break

    hands, img = detector.findHands(img)

    # =========================
    # [7] Keyboard Input
    # =========================
    key = cv2.waitKey(1) & 0xFF

    # Select label (A-Z)
    if 65 <= key <= 90:
        current_label = chr(key)
        print(f"Selected Label: {current_label}")

    # Select number (0-9)
    elif 48 <= key <= 57:
        if mode == "number":
            current_label = chr(key)
            print(f"Selected Label: {current_label}")

    # Start collecting
    elif key == ord('s') and current_label is not None:
        collecting = True
        sequence = []
        print(f"Collecting for: {current_label}")

    # Quit
    elif key == 27:
        break

    # =========================
    # [8] Hand Processing
    # =========================
    if hands:
        hand = hands[0]

        # 🔥 Extract landmarks
        lmList = np.array(hand['lmList'])

        # =========================
        # [9] Feature Engineering (IMPORTANT)
        # =========================
        # Make wrist (point 0) as origin
        lmList = lmList - lmList[0]

        # Flatten → (63,)
        feature_vector = lmList.flatten()

        # =========================
        # [10] Sequence Collection
        # =========================
        if collecting:
            sequence.append(feature_vector)

            print(f"Collecting: {len(sequence)}/{sequence_length}")

            if len(sequence) == sequence_length:
                # =========================
                # [11] Save Data
                # =========================
                save_dir = f"data/raw/{mode}/{current_label}"
                count = len(os.listdir(save_dir))

                save_path = os.path.join(save_dir, f"{count}.npy")

                np.save(save_path, np.array(sequence))

                print(f"Saved: {save_path}")

                # Reset
                sequence = []
                collecting = False

    # =========================
    # [12] Display Info (UI)
    # =========================
    cv2.putText(img, f"Mode: {mode}", (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    cv2.putText(img, f"Label: {current_label}", (10, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.putText(img, f"Collecting: {collecting}", (10, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # =========================
    # [13] Show Frame
    # =========================
    cv2.imshow("Image", img)

# =========================
# [14] Release Resources
# =========================
cap.release()
cv2.destroyAllWindows()